In [1]:
import torch
from torch_geometric.data import HeteroData
from torch_geometric.loader import NeighborLoader

import pandas as pd
import numpy as np

import os
import json

from utils.hake_dataset import *

from hake.models import HAKE
from hake.data import DataReader, TrainDataset, BatchType, BidirectionalOneShotIterator
from torch.utils.data import DataLoader

import warnings
warnings.filterwarnings('ignore')

/home/mai/miniconda3/envs/pyg-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_PATH = './data'

# folder to save entity id mapping
ID_MAPPING = os.path.join(DATA_PATH, 'entity_id_map')
os.makedirs(ID_MAPPING, exist_ok=True)

# folder to save edge index
EDGE_INDEX = os.path.join(DATA_PATH, 'edge_index')
os.makedirs(EDGE_INDEX, exist_ok=True)

# folder to save graph data
GRAPH_DATA = os.path.join(DATA_PATH, 'graph_data')
os.makedirs(GRAPH_DATA, exist_ok=True)

# folder to save HAKE data and model
HAKE_PATH = './hake'
os.makedirs(HAKE_PATH, exist_ok=True)

HAKE_DATA = os.path.join(HAKE_PATH, 'data/yfinance_kge')
os.makedirs(HAKE_DATA, exist_ok=True)

HAKE_EMBEDDINGS = os.path.join(HAKE_PATH, 'embeddings/yfinance_kge')
os.makedirs(HAKE_EMBEDDINGS, exist_ok=True)

In [3]:
data = torch.load(os.path.join(GRAPH_DATA, 'yfinance_kge.pt'))

In [4]:
data

HeteroData(
  company={ x=[4164, 38] },
  fund_symbol={ x=[6340, 29] },
  sector={ x=[13, 7] },
  industry={ x=[146, 9] },
  (company, has_symbol, stock_symbol)={ edge_index=[2, 4508] },
  (company, belongs_to, industry)={ edge_index=[2, 3992] },
  (industry, is_part_of, sector)={ edge_index=[2, 144] },
  (institution, holds, stock_symbol)={ edge_index=[2, 39512] },
  (fund, holds, stock_symbol)={ edge_index=[2, 37098] },
  (fund, has_symbol, fund_symbol)={ edge_index=[2, 6851] }
)

# 1. Create data input for HAKE model

In [5]:
# prepare entities.dict and relations.dict
entity2gid = create_hake_dicts(hetero_data=data, out_dir=HAKE_DATA)

# save entity2gid to later map HAKE embeddings to graph
with open(os.path.join(ID_MAPPING, 'entity2gid.json'), 'w') as f:
    json.dump(entity2gid, f, indent=2)

In [6]:
# create train, valid, test data txt files
create_hake_txt_files(data, num_val=0.4, num_test=0.4, out_dir=HAKE_DATA)

# 2. Train HAKE to generate node embeddings

In [7]:
data_reader = DataReader(HAKE_DATA)

num_entity = len(data_reader.entity_dict)
num_relation = len(data_reader.relation_dict)

print(f"Num entities: {num_entity}")
print(f"Num relations: {num_relation}")

Num entities: 20050
Num relations: 4


In [8]:
MODEL = 'HAKE'
HIDDEN_DIM = 16
GAMMA = 12.0
BATCH_SIZE = 128
NEGATIVE_SAMPLE_SIZE = 128
LEARNING_RATE = 0.0001
MAX_STEPS = 1000
MODULUS_WEIGHT = 1.0
PHASE_WEIGHT = 0.5
ADVERSARIAL_TEMPERATURE = 1.0
CPU_NUM = 4

In [9]:
# -----------------------------
# Model
# -----------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = HAKE(num_entity, num_relation, HIDDEN_DIM, GAMMA, MODULUS_WEIGHT, PHASE_WEIGHT).to(device)

In [10]:
# -----------------------------
# DataLoader & iterator
# -----------------------------
train_dataloader_head = DataLoader(
    TrainDataset(data_reader, NEGATIVE_SAMPLE_SIZE, BatchType.HEAD_BATCH),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=CPU_NUM,
    collate_fn=TrainDataset.collate_fn
)

train_dataloader_tail = DataLoader(
    TrainDataset(data_reader, NEGATIVE_SAMPLE_SIZE, BatchType.TAIL_BATCH),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=CPU_NUM,
    collate_fn=TrainDataset.collate_fn
)

train_iterator = BidirectionalOneShotIterator(train_dataloader_head, train_dataloader_tail)

In [11]:
# -----------------------------
# Optimizer
# -----------------------------
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [12]:
# -----------------------------
# Training loop
# -----------------------------
for step in range(MAX_STEPS):
    log = model.train_step(model, optimizer, train_iterator, args=type('args', (), {'adversarial_temperature': ADVERSARIAL_TEMPERATURE}))
    
    if step % 200 == 0:
        print(f"Step {step}: loss={log['loss']:.4f}")

Step 0: loss=2.5938
Step 200: loss=2.4491
Step 400: loss=2.3347
Step 600: loss=2.2002
Step 800: loss=2.0445


In [13]:
# -----------------------------
# Save checkpoint
# -----------------------------
HAKE_CHECKPOINT = os.path.join(HAKE_PATH, 'checkpoints')
os.makedirs(HAKE_CHECKPOINT, exist_ok=True)

checkpoint = {
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "num_entity": num_entity,
    "num_relation": num_relation,
    "hidden_dim": HIDDEN_DIM,
    "gamma": GAMMA,
    "modulus_weight": MODULUS_WEIGHT,
    "phase_weight": PHASE_WEIGHT
}

# full checkpoint
torch.save(checkpoint, os.path.join(HAKE_CHECKPOINT, "checkpoint.pt"))

# save config for references
config = {
    "MODEL": MODEL,
    "HIDDEN_DIM": HIDDEN_DIM,
    "GAMMA": GAMMA,
    "MODULUS_WEIGHT": MODULUS_WEIGHT,
    "PHASE_WEIGHT": PHASE_WEIGHT,
    "NEGATIVE_SAMPLE_SIZE": NEGATIVE_SAMPLE_SIZE,
    "LEARNING_RATE": LEARNING_RATE,
    "MAX_STEPS": MAX_STEPS,
    "BATCH_SIZE": BATCH_SIZE
}
with open(os.path.join(HAKE_CHECKPOINT, "config.json"), "w") as f:
    json.dump(config, f, indent=2)

print("✓ Model checkpoint saved to", HAKE_CHECKPOINT)

✓ Model checkpoint saved to ./hake/checkpoints


In [14]:
# -----------------------------
# Save embeddings
# -----------------------------
torch.save(model.entity_embedding.detach().cpu(), os.path.join(HAKE_EMBEDDINGS, 'entity_embedding.pt'))
torch.save(model.relation_embedding.detach().cpu(), os.path.join(HAKE_EMBEDDINGS, 'relation_embedding.pt'))
print("Embeddings saved!")

Embeddings saved!


In [15]:
# # -----------------------------
# # Load checkpoint
# # -----------------------------
# checkpoint_path = os.path.join(HAKE_CHECKPOINT, "checkpoint.pt")
# checkpoint = torch.load(checkpoint_path, map_location="cpu")

In [16]:
# # Rebuild model from checkpoint info
# loaded_model = HAKE(
#     num_entity=checkpoint["num_entity"],
#     num_relation=checkpoint["num_relation"],
#     hidden_dim=checkpoint["hidden_dim"],
#     gamma=checkpoint["gamma"],
#     modulus_weight=checkpoint["modulus_weight"],
#     phase_weight=checkpoint["phase_weight"]
# )

# loaded_model.load_state_dict(checkpoint["model_state_dict"])
# loaded_model.eval()

# print("✓ Model restored and ready.")

In [17]:
# loaded_model.entity_embedding

# 3. Add HAKE embeddings to the hetero graph created
Hake embeddings for entities uses the index in the `entities.dict` file so we'll have to map it to our original id mapping

In [18]:
# Load HAKE embeddings (shape [num_entities, dim])
emb = torch.load(os.path.join(HAKE_EMBEDDINGS, 'entity_embedding.pt')).cpu().numpy()

dim = emb.shape[1]

# load entity name: global id mapping of the heterogeneous graph
with open(os.path.join(ID_MAPPING, 'entity2gid.json')) as f:
    entity2gid = json.load(f)

num_global = max(entity2gid.values()) + 1

# load entities.dict created for HAKE embedding
def load_entities_dict(path):
    entity2id = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            eid, name = line.strip().split("\t")
            entity2id[name] = int(eid)
    return entity2id

hake_entities = load_entities_dict(os.path.join(HAKE_DATA, 'entities.dict'))

# Your final embedding matrix aligned to the graph
gid_emb = np.zeros((num_global, dim))

for name, gid in entity2gid.items():
    hake_id = hake_entities[name]  # from entities.dict OR your code
    gid_emb[gid] = emb[hake_id]


In [19]:
gid_emb.shape

(20161, 32)

In [20]:
gid_emb

array([[ 0.10306585,  0.47510025,  0.60801727, ...,  0.11677513,
         0.53720337, -0.46682924],
       [-0.64783156, -0.13978666,  0.39654174, ...,  0.07680555,
         0.20179678,  0.3280898 ],
       [-0.08063221,  0.46182081, -0.29783705, ..., -0.88061744,
        -0.07311786,  0.84099352],
       ...,
       [ 0.38793948, -0.57555819, -0.76737463, ...,  0.85134679,
        -0.28404292,  0.18882141],
       [-0.28707212, -0.81313777,  0.0575695 , ..., -0.77826226,
        -0.24450791,  0.43440667],
       [ 0.74198461, -0.52178729, -0.74333739, ..., -0.39157122,
        -0.15026617,  0.49185807]])

In [21]:
# load heterogeneous graph created so far
data = torch.load(os.path.join(GRAPH_DATA, 'yfinance_kge.pt'), weights_only=False)

# convert gid_emb to torch tensor
gid_emb_tensor = torch.tensor(gid_emb, dtype=torch.float)

# load node id offsets
with open(os.path.join(ID_MAPPING, 'global_entity_id_offsets.json')) as f:
    offset_start = json.load(f)

# compute the start and end id of each entity types
sorted_types = sorted(offset_start.items(), key=lambda x: x[1])

offsets = {}

for i, (ntype, start) in enumerate(sorted_types):
    if i < len(sorted_types) - 1:
        _, next_start = sorted_types[i+1]
        end = next_start
    else:
        end = gid_emb.shape[0]
    offsets[ntype] = (start, end)

# Now concat HAKE embeddings to the node features in HeteroData
gid_emb_tensor = torch.tensor(gid_emb, dtype=torch.float)

for ntype, (start, end) in offsets.items():
    node_emb = gid_emb_tensor[start:end]
    
    if 'x' in data[ntype]:
        data[ntype].x = torch.cat([data[ntype].x, node_emb], dim=1)
    else:
        data[ntype].x = node_emb

    print(f"{ntype}: {data[ntype].x.shape}")

company: torch.Size([4164, 70])
stock_symbol: torch.Size([4607, 32])
industry: torch.Size([146, 41])
sector: torch.Size([13, 39])
institution: torch.Size([2669, 32])
fund: torch.Size([2222, 32])
fund_symbol: torch.Size([6340, 61])


In [22]:
data

HeteroData(
  company={ x=[4164, 70] },
  fund_symbol={ x=[6340, 61] },
  sector={ x=[13, 39] },
  industry={ x=[146, 41] },
  stock_symbol={ x=[4607, 32] },
  institution={ x=[2669, 32] },
  fund={ x=[2222, 32] },
  (company, has_symbol, stock_symbol)={ edge_index=[2, 4508] },
  (company, belongs_to, industry)={ edge_index=[2, 3992] },
  (industry, is_part_of, sector)={ edge_index=[2, 144] },
  (institution, holds, stock_symbol)={ edge_index=[2, 39512] },
  (fund, holds, stock_symbol)={ edge_index=[2, 37098] },
  (fund, has_symbol, fund_symbol)={ edge_index=[2, 6851] }
)

In [23]:
# save graph with KGE embeddings and original features
torch.save(data, os.path.join(GRAPH_DATA, 'yfinance_kge_full.pt'))